# Optimización de un rango de *push/fold* en póker heads-up con un Algoritmo Genético

**Algoritmos Evolutivos I (2026) — Maestría en Inteligencia Artificial, FIUBA**  
Desafío Práctico · Técnica utilizada: **Algoritmos Genéticos (GA)**

> **Repositorio con el código fuente:** https://github.com/Nestiii/AE1  
> *(esta misma notebook está en `notebooks/`).*

Esta notebook es autocontenida y reproducible en **Google Colab**: clona el
repositorio, importa el código de `src/pushfold/` y reproduce todos los
resultados y figuras del trabajo.

## 1. El problema

En **Texas Hold'em heads-up** (1 contra 1) con **stacks cortos**, la teoría de
juegos muestra que la estrategia óptima del botón / ciega chica (SB) se reduce a
una decisión binaria: **ir all-in (push) o retirarse (fold)** antes del flop.
Jugar 'push o fold' evita todas las decisiones posteriores (flop, turn, river),
que con stacks cortos aportan poco valor y sí mucho riesgo.

El jugador debe decidir, para **cada una de las 169 manos iniciales** distintas
(13 pares + 78 *suited* + 78 *offsuit*), si la pushea o la foldea. Esa decisión
conjunta es un **rango de push**, y hay $2^{169}$ rangos posibles: un espacio de
búsqueda enorme, ideal para una metaheurística.

**Por qué un Algoritmo Genético:** el rango se codifica naturalmente como un
**vector binario de 169 bits** (1 = push, 0 = fold), que es exactamente el
cromosoma de un GA binario clásico. El *fitness* es la ganancia esperada (en
*big blinds* por mano) que ese rango obtiene contra el rival, estimada con un
modelo de valor esperado sobre equities reales de póker.

## 2. Preparación del entorno

Clonamos el repositorio (en Colab) e importamos el paquete `pushfold`. El repo
incluye la **matriz de equity ya precomputada** (`data/equity_169x169.npy`), así
que no hace falta compilar `eval7`: alcanza con `numpy` y `matplotlib`.

In [ ]:
import os, sys, subprocess

# URL del repositorio con el código fuente.
REPO_URL = "https://github.com/Nestiii/AE1.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("AE1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("AE1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib"], check=True)

# Nos ubicamos en la raíz del repo (carpeta que contiene src/pushfold),
# subiendo directorios si hiciera falta (p.ej. al correr desde notebooks/).
_d = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_d, "src", "pushfold")):
        os.chdir(_d)
        break
    _d = os.path.dirname(_d)

# Aseguramos que src/ esté en el path para importar el paquete.
sys.path.insert(0, os.path.abspath("src"))
print("Directorio de trabajo:", os.getcwd())
print("¿En Colab?:", IN_COLAB)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pushfold.equity import get_equity_matrix
from pushfold.game import PushFoldGame
from pushfold.ga import GAConfig, run_ga
from pushfold.plots import convergence_plot, boxplot_by_budget, range_grid_plot

# Carga la matriz de equity versionada (o la regenera con eval7 si no existe).
equity = get_equity_matrix()
print("Matriz de equity:", equity.shape)

## 3. Modelo del juego y función de *fitness*

Con un stack efectivo de $S$ *big blinds* (BB), ciega chica $0.5$ y ciega grande
$1$, el valor esperado (EV) de la SB en cada resultado es:

| Acción de la SB | Resultado | EV (BB) |
|---|---|---|
| Fold | pierde su ciega chica | $-0.5$ |
| Push y la BB foldea | se lleva la ciega grande | $+1.0$ |
| Push y la BB paga | showdown all-in | $S\,(2\,eq - 1)$ |

donde $eq$ es la *equity* (probabilidad de ganar) del héroe contra la mano del
villano. Para cada mano $h$ del héroe, promediando sobre el rango con el que la
BB paga (fijo en esta versión):

$$EV_{push}(h) = \sum_j w_j\left[c_j\,S(2\,eq_{hj}-1) + (1-c_j)\cdot 1\right],\qquad EV_{fold} = -0.5$$

y el *fitness* de un cromosoma $x$ (ganancia esperada por mano) es

$$\text{fitness}(x) = \sum_h w_h\left[x_h\,EV_{push}(h) + (1-x_h)\,EV_{fold}\right].$$

Como el rango de la BB es fijo, $EV_{push}(h)$ no depende de las demás manos: el
problema es **separable** y su óptimo se conoce en forma cerrada (pushear si y
solo si $EV_{push}(h) > EV_{fold}$). Lo usamos como **verdad de referencia** para
validar que el GA converge al óptimo correcto.

In [ ]:
STACK_BB = 10.0        # stack efectivo en big blinds
BB_CALL_FRACTION = 0.5 # la BB paga con el 50% de manos más fuertes (rival fijo)

game = PushFoldGame(equity=equity, stack_bb=STACK_BB,
                    bb_call_fraction=BB_CALL_FRACTION)

opt_genome = game.analytical_optimum()
opt_fit = game.optimum_fitness()
print(f"Óptimo analítico: fitness = {opt_fit:.5f} BB/mano")
print(f"Pushea el {game.push_percentage(opt_genome):.1f}% de las manos")

## 4. El Algoritmo Genético

Codificación y operadores clásicos de un GA binario:

- **Cromosoma:** 169 bits (1 = push, 0 = fold).
- **Selección por torneo** (tamaño 3).
- **Cruza uniforme** (cada gen se hereda de uno u otro padre).
- **Mutación bit-flip** (probabilidad $\approx 1/169$ por gen).
- **Elitismo** (los 2 mejores pasan intactos → el mejor *fitness* nunca empeora).

Corremos el GA y graficamos la **curva de convergencia** (entregable obligatorio
de la consigna): mejor y promedio de la población por generación, con el óptimo
analítico de referencia.

In [ ]:
config = GAConfig(pop_size=80, n_generations=120,
                  tournament_size=3, elitism=2, seed=0)
result = run_ga(game.fitness_population, config, verbose=True)

print(f"\nMejor fitness del GA: {result.best_fitness:.5f} BB/mano")
n_diff = int(np.sum(result.best_genome != opt_genome))
print(f"Diferencias con el óptimo analítico: {n_diff}/169 manos")

In [ ]:
convergence_plot(result.best_history, result.mean_history, optimum=opt_fit)
plt.tight_layout(); plt.show()

## 5. Robustez: diagrama de caja por presupuesto de generaciones

Repetimos el GA con **30 semillas** independientes. Como el mejor *fitness* es
monótono (por el elitismo), el historial de una corrida nos da su *fitness* final
para cualquier presupuesto de generaciones. El diagrama de caja muestra cómo, al
aumentar el presupuesto, la mediana sube hacia el óptimo y la **dispersión entre
corridas se reduce**: el GA es cada vez más confiable.

In [ ]:
N_RUNS = 30
histories = []
for r in range(N_RUNS):
    res = run_ga(game.fitness_population,
                 GAConfig(pop_size=80, n_generations=120, seed=r))
    histories.append(np.asarray(res.best_history))
histories = np.vstack(histories)

budgets = [15, 30, 60, 120]
fitness_by_budget = [histories[:, b].tolist() for b in budgets]
boxplot_by_budget(fitness_by_budget, budgets, optimum=opt_fit)
plt.tight_layout(); plt.show()

## 6. El rango de push evolucionado

Finalmente visualizamos el cromosoma ganador como la clásica **grilla 13×13** de
manos de póker (verde = push, gris = fold). Las celdas que difieran del óptimo
analítico se marcarían con borde rojo (no debería haber ninguna).

In [ ]:
range_grid_plot(result.best_genome, reference=opt_genome,
                title=f'Rango de push evolucionado (stack {STACK_BB:.0f} BB)')
plt.tight_layout(); plt.show()

print("Manos que se pushean ({:.1f}% del total):".format(game.push_percentage(result.best_genome)))
print(', '.join(game.range_labels(result.best_genome)))

## 7. Conclusiones, inconvenientes y trabajo futuro

**Resultados.** El GA converge de forma estable al **óptimo analítico exacto**
(0/169 manos de diferencia) y recupera un rango de push con sentido pokerístico:
todos los pares, todos los ases, reyes *suited*, *broadways* y conectores. La
curva de convergencia y el diagrama de caja confirman la fiabilidad del método.

**Inconvenientes encontrados y cómo se resolvieron:**

1. *Costo de calcular las equities.* Estimar la *equity* de las 169×169
   combinaciones por Monte Carlo en Python puro era demasiado lento. Se resolvió
   usando el evaluador en C `eval7` (`py_hand_vs_range_monte_carlo`) y
   **precomputando la matriz una sola vez** (≈25 s), que se versiona en el repo.
2. *Warnings espurios de `matmul`* en macOS (Apple Accelerate + numpy 2.0). Se
   verificó que el resultado era correcto y se reemplazó el producto matriz-vector
   por una reducción *elementwise* equivalente, robusta en cualquier backend.
3. *Diagrama de caja degenerado.* Al ser el problema separable, todas las corridas
   alcanzaban el óptimo exacto y el box plot colapsaba a un punto. Se replanteó
   para mostrar el *fitness* final **a distintos presupuestos de generaciones**,
   que sí exhibe dispersión y comunica la velocidad de convergencia.

**Trabajo futuro (extensión A2).** El rival (rango de call de la BB) es fijo, lo
que hace el problema separable. Una extensión natural es **coevolucionar** los
rangos de push de la SB y de call de la BB (juego de suma cero) para aproximar el
**equilibrio de Nash** de push/fold; ahí el problema deja de ser separable y el GA
cobra un rol más interesante como buscador de equilibrios.